# ATTS-RAG: GPU runs for reviewer points 1, 4 and 5 (free Kaggle GPU)

This notebook runs the LLM-heavy and NLI-heavy experiments that are too slow on the author's CPU. Everything is free and local to the notebook. No paid API is used, and every evaluator is an open model.

| Job | Reviewer point | What it produces |
|---|---|---|
| RAGTruth, small and large NLI | 1, 4, 5 | Layer 3 verifier on 2,675 human-labelled answers, with `nli-deberta-v3-small` and `DeBERTa-v3-large` |
| Llama 3, small vs large NLI | 5 | Dev + held-out quality with each NLI model, same Llama 3 drafts |
| More generators | 5 | Gemma 4 E4B, Mistral 7B and Phi-4 14B through the same pipeline (dev + held-out) |
| Independent judge on each generator | 5 | qwen2.5:14b labels every answer sentence (another model family) |
| Judge vs human labels | 5 | qwen2.5:14b against RAGTruth's human labels (Cohen's kappa) |

**Before running** (right-hand panel → *Settings*):
1. **Accelerator:** *GPU T4 x2* (or *GPU P100*).
2. **Internet:** *On*. Kaggle asks for a verified phone number the first time.

**To run:** *Save Version* → *Save & Run All (Commit)*. It runs in the background (the browser can be closed) for about 8–10 hours. Then open the version's **Output** tab and download `atts_gpu_results.zip`.

Jobs that would start after `TIME_LIMIT_H` are skipped, so the results are always saved before Kaggle's 12-hour limit. `SMOKE = True` runs a few questions per job (about 30 minutes), to check the setup.

In [1]:
# ---- settings ------------------------------------------------------------------------------------
REPO = "https://github.com/SakthiQ/ATTS-RAG.git"
BRANCH = "claude/layer1-detection"
GENERATORS = ["gemma4:e4b", "mistral:7b", "phi4:14b"]   # Llama 3 (the main generator) is run by the llama3 jobs
JUDGE = "qwen2.5:14b"                                     # independent evaluator, a different family from every generator
NLI_LARGE = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
JOBS = {
    "ragtruth": True,          # verifier on RAGTruth with the small and the large NLI model (runs in the background)
    "llama3_nli": True,        # Llama 3 dev + held-out quality, small vs large NLI
    "generators": True,        # the GENERATORS through the quality suite
    "judge_generators": True,  # the judge labels every sentence of each generator
    "judge_ragtruth": True,    # the judge against RAGTruth's human labels
}
TIME_LIMIT_H = 11.0   # Kaggle stops a session at 12 h
SMOKE = False         # True: a few questions per job, to check the setup

In [2]:
# ---- setup: code, packages, benchmark, Ollama ------------------------------------------------------
import json, os, shutil, subprocess, sys, time, urllib.request

T0 = time.time()
BASE = "/tmp/atts"
REPO_DIR = f"{BASE}/ATTS-RAG"
RT_DATA = f"{BASE}/ragtruth/dataset"
OUT = "/kaggle/working/atts_gpu" if os.path.isdir("/kaggle/working") else f"{BASE}/out"
R = f"{OUT}/results"
CACHE = f"{OUT}/llm_cache.jsonl"
HOST = "http://127.0.0.1:11434"
for d in (BASE, OUT, R, f"{OUT}/logs"):
    os.makedirs(d, exist_ok=True)
STATUS = {}


def say(*a):
    print(f"[{(time.time() - T0) / 3600:4.1f} h]", *a, flush=True)


def sh(cmd, log=None, cwd=None, env=None):
    """Runs a shell command; output goes to `log` (or is printed). Returns the exit code."""
    out = open(log, "a") if log else None
    rc = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=out, stderr=subprocess.STDOUT if out else None).returncode
    if out:
        out.close()
    return rc


def left_h():
    return TIME_LIMIT_H - (time.time() - T0) / 3600


sh("nvidia-smi -L")
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
say(f"{NGPU} GPU(s)")
assert NGPU > 0, "No GPU: set Accelerator to GPU in the notebook settings"

if not os.path.isdir(REPO_DIR):
    assert sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR}") == 0, "git clone failed: is Internet on?"
COMMIT = subprocess.run(f"git -C {REPO_DIR} rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
say("ATTS-RAG commit", COMMIT)
if not os.path.isdir(RT_DATA):
    sh(f"git clone -q --depth 1 https://github.com/ParticleMedia/RAGTruth.git {BASE}/ragtruth")

PKGS = ("langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters langchain-ollama "
        "chromadb tiktoken rank_bm25 loguru tenacity pypdf pdfplumber pymupdf python-docx rapidocr-onnxruntime "
        "python-dotenv emoji langdetect rapidfuzz")
assert sh(f"{sys.executable} -m pip install -q -U {PKGS}", log=f"{OUT}/logs/pip.log") == 0, "pip install failed, see logs/pip.log"

if shutil.which("ollama") is None:
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", log=f"{OUT}/logs/apt.log")
    assert sh("curl -fsSL https://ollama.com/install.sh | sh", log=f"{OUT}/logs/ollama_install.log") == 0, "Ollama install failed"
OLLAMA_ENV = dict(os.environ, OLLAMA_MODELS=f"{BASE}/ollama", OLLAMA_KEEP_ALIVE="30m", OLLAMA_HOST="127.0.0.1:11434")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV, stdout=open(f"{OUT}/logs/ollama_serve.log", "a"),
                               stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(HOST + "/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
say("Ollama up")

# Llama 3 drafts from the main (CPU) runs: the llama3 jobs reuse them, so their results pair with the main results
snap = f"{REPO_DIR}/eval/cache_snapshot/llm_cache.jsonl"
if os.path.exists(snap) and not os.path.exists(CACHE):
    shutil.copy(snap, CACHE)
say("LLM cache entries:", sum(1 for _ in open(CACHE)) if os.path.exists(CACHE) else 0)

ENV = dict(os.environ, PYTHONPATH=".", HF_HOME=f"{BASE}/hf", TOKENIZERS_PARALLELISM="false",
           JUDGE_OLLAMA_HOST=HOST, OLLAMA_HOST="127.0.0.1:11434")
LIM_Q = "--limit 3" if SMOKE else ""       # quality suite: first N answerable questions
LIM_RT = "--limit 5" if SMOKE else ""      # RAGTruth: first N answers per task
N_JUDGE_RT = 10 if SMOKE else 300
HELDOUT = "--qa-file eval/qa_heldout.jsonl --extra-corpus eval/corpus_heldout"


def save():
    """Writes the job status and zips everything, after every job, so a stopped session keeps what finished."""
    json.dump({"commit": COMMIT, "smoke": SMOKE, "gpus": NGPU, "hours": round((time.time() - T0) / 3600, 2), "jobs": STATUS},
              open(f"{OUT}/status.json", "w"), indent=1)
    if os.path.isdir("/kaggle/working"):
        shutil.make_archive("/kaggle/working/atts_gpu_results", "zip", OUT)


def job(name, cmd, env=None, need_h=0.3):
    if left_h() < need_h:
        STATUS[name] = "skipped (time limit)"
        say("SKIP", name, "(time limit)")
        return False
    say("START", name)
    t = time.time()
    log = f"{OUT}/logs/{name}.log"
    rc = sh(cmd, log=log, cwd=REPO_DIR, env=dict(ENV, **(env or {})))
    STATUS[name] = f"{'ok' if rc == 0 else 'FAILED (exit ' + str(rc) + ')'} in {(time.time() - t) / 60:.0f} min"
    say(STATUS[name].upper() if rc else STATUS[name], name)
    if rc:
        print("".join(open(log, errors="replace").readlines()[-15:]))
    save()
    return rc == 0


def background(name, cmd, env=None):
    say("START (background)", name)
    return subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, env=dict(ENV, **(env or {})),
                            stdout=open(f"{OUT}/logs/{name}.log", "a"), stderr=subprocess.STDOUT)


def pull(model):
    say("pull", model)
    return sh(f"ollama pull {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV) == 0


def drop(model):
    sh(f"ollama rm {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV)


def tag(model):
    return model.replace(":", "-").replace(".", "")


save()

GPU 0: Tesla T4 (UUID: GPU-0e0ae0a6-6654-680f-3635-f367732494a7)
GPU 1: Tesla T4 (UUID: GPU-56a404d0-84e6-9717-8016-066b682593dd)
[ 0.0 h] 2 GPU(s)
[ 0.0 h] ATTS-RAG commit e687c41
[ 0.0 h] Ollama up
[ 0.0 h] LLM cache entries: 0


In [3]:
# ---- 1. RAGTruth: the Layer 3 verifier with the small and the large NLI model (background, one GPU each) ---------
BG = {}
if JOBS["ragtruth"]:
    for i, (name, nli) in enumerate([("ragtruth_nli_small", ""), ("ragtruth_nli_large", NLI_LARGE)]):
        env = {"CUDA_VISIBLE_DEVICES": str(i % NGPU)}
        if nli:
            env["NLI_MODEL"] = nli
        BG[name] = background(name, f"python -u scripts/eval_ragtruth.py --data {RT_DATA} --out {R}/{name} --threads 2 {LIM_RT}", env)
        BG[name].t0 = time.time()

[ 0.0 h] START (background) ragtruth_nli_small
[ 0.0 h] START (background) ragtruth_nli_large


In [4]:
# ---- 2. Llama 3: dev + held-out quality, small vs large NLI (same drafts) --------------------------------
# The drafts come from the cache of the main CPU runs; the report's "LLM calls" line says how many had to be regenerated.
if JOBS["llama3_nli"]:
    pull("llama3")
    for split, extra in (("dev", ""), ("heldout", HELDOUT)):
        for nli_name, nli in (("small", ""), ("large", NLI_LARGE)):
            job(f"llama3_{split}_nli_{nli_name}",
                f"python -u scripts/evaluate_pipeline.py --suite quality --model llama3 --cache {CACHE} {extra} {LIM_Q} "
                f"--out {R}/quality_{split}_llama3_nli_{nli_name}", env={"NLI_MODEL": nli} if nli else None)
    drop("llama3")

[ 0.0 h] pull llama3
[ 0.1 h] START llama3_dev_nli_small
[ 0.2 h] ok in 9 min llama3_dev_nli_small
[ 0.2 h] START llama3_dev_nli_large
[ 0.2 h] ok in 2 min llama3_dev_nli_large
[ 0.2 h] START llama3_heldout_nli_small
[ 0.4 h] ok in 7 min llama3_heldout_nli_small
[ 0.4 h] START llama3_heldout_nli_large
[ 0.4 h] ok in 1 min llama3_heldout_nli_large


In [5]:
# ---- 3. More generators: dev + held-out quality, then the sentences the judge will label ------------------------
if JOBS["generators"]:
    for m in GENERATORS:
        need = 2.0 if "14b" in m else 1.2
        if left_h() < need:
            STATUS[f"generator {m}"] = "skipped (time limit)"
            say("SKIP generator", m)
            continue
        if not pull(m):
            STATUS[f"generator {m}"] = "FAILED (ollama pull)"
            continue
        t = tag(m)
        job(f"quality_dev_{t}", f"python -u scripts/evaluate_pipeline.py --suite quality --model {m} --cache {CACHE} {LIM_Q} --out {R}/quality_dev_{t}")
        job(f"quality_heldout_{t}", f"python -u scripts/evaluate_pipeline.py --suite quality --model {m} --cache {CACHE} {HELDOUT} {LIM_Q} --out {R}/quality_heldout_{t}")
        job(f"claims_{t}", f"python -u scripts/independent_judge.py claims --model {m} --heldout --cache {CACHE} --out {R}/indep_{t}")
        drop(m)

[ 0.4 h] pull gemma4:e4b
[ 0.4 h] START quality_dev_gemma4-e4b
[ 0.5 h] ok in 4 min quality_dev_gemma4-e4b
[ 0.5 h] START quality_heldout_gemma4-e4b
[ 0.5 h] ok in 3 min quality_heldout_gemma4-e4b
[ 0.5 h] START claims_gemma4-e4b
[ 0.5 h] ok in 1 min claims_gemma4-e4b
[ 0.5 h] pull mistral:7b
[ 0.5 h] START quality_dev_mistral-7b
[ 0.6 h] ok in 4 min quality_dev_mistral-7b
[ 0.6 h] START quality_heldout_mistral-7b
[ 0.7 h] ok in 5 min quality_heldout_mistral-7b
[ 0.7 h] START claims_mistral-7b
[ 0.7 h] ok in 1 min claims_mistral-7b
[ 0.7 h] pull phi4:14b
[ 0.7 h] START quality_dev_phi4-14b
[ 0.9 h] ok in 9 min quality_dev_phi4-14b
[ 0.9 h] START quality_heldout_phi4-14b
[ 1.0 h] ok in 10 min quality_heldout_phi4-14b
[ 1.0 h] START claims_phi4-14b
[ 1.0 h] ok in 1 min claims_phi4-14b


In [6]:
# ---- 4. Independent judge: every sentence of each generator, and RAGTruth sentences against human labels --------
if JOBS["judge_generators"] or JOBS["judge_ragtruth"]:
    pull(JUDGE)
if JOBS["judge_generators"]:
    for m in GENERATORS:
        t = tag(m)
        if os.path.exists(f"{R}/indep_{t}/claims.jsonl"):
            if job(f"judge_{t}", f"python -u scripts/independent_judge.py judge --out {R}/indep_{t} --host {HOST} --judge-model {JUDGE}", need_h=0.5):
                job(f"judge_report_{t}", f"python -u scripts/independent_judge.py report --out {R}/indep_{t}", need_h=0.05)
if JOBS["judge_ragtruth"] and "ragtruth_nli_small" in BG:
    p = BG["ragtruth_nli_small"]
    while p.poll() is None and left_h() > 1.0:   # the sample is drawn from the small-NLI RAGTruth run
        time.sleep(60)
    if p.poll() is not None:
        job("judge_ragtruth", f"python -u scripts/judge_ragtruth.py --n {N_JUDGE_RT} --data {RT_DATA} --items {R}/ragtruth_nli_small/items.jsonl "
            f"--out {R}/judge_ragtruth --host {HOST} --judge-model {JUDGE}", need_h=0.5)
    else:
        STATUS["judge_ragtruth"] = "skipped (RAGTruth small not finished in time)"

[ 1.0 h] pull qwen2.5:14b
[ 1.0 h] START judge_gemma4-e4b
[ 1.3 h] ok in 14 min judge_gemma4-e4b
[ 1.3 h] START judge_report_gemma4-e4b
[ 1.3 h] ok in 0 min judge_report_gemma4-e4b
[ 1.3 h] START judge_mistral-7b
[ 1.4 h] ok in 9 min judge_mistral-7b
[ 1.4 h] START judge_report_mistral-7b
[ 1.4 h] ok in 0 min judge_report_mistral-7b
[ 1.4 h] START judge_phi4-14b
[ 1.6 h] ok in 13 min judge_phi4-14b
[ 1.6 h] START judge_report_phi4-14b
[ 1.6 h] ok in 0 min judge_report_phi4-14b
[ 1.6 h] START judge_ragtruth
[ 2.0 h] ok in 19 min judge_ragtruth


In [7]:
# ---- 5. Wait for RAGTruth, write its reports, save everything ------------------------------------------------
for name, p in BG.items():
    while p.poll() is None and left_h() > 0.15:
        time.sleep(60)
    if p.poll() is None:
        p.terminate()
        p.wait()
        STATUS[name] = "stopped at the time limit (partial; items.jsonl is resumable)"
    else:
        STATUS[name] = f"{'ok' if p.returncode == 0 else 'FAILED (exit ' + str(p.returncode) + ')'} in {(time.time() - p.t0) / 60:.0f} min"
    if os.path.exists(f"{R}/{name}/items.jsonl"):
        sh(f"python -u scripts/eval_ragtruth.py --data {RT_DATA} --out {R}/{name} --report-only", log=f"{OUT}/logs/{name}.log",
           cwd=REPO_DIR, env=ENV)
save()
say("DONE")
for k, v in STATUS.items():
    print(f"  {k:32s} {v}")

[ 2.0 h] DONE
  llama3_dev_nli_small             ok in 9 min
  llama3_dev_nli_large             ok in 2 min
  llama3_heldout_nli_small         ok in 7 min
  llama3_heldout_nli_large         ok in 1 min
  quality_dev_gemma4-e4b           ok in 4 min
  quality_heldout_gemma4-e4b       ok in 3 min
  claims_gemma4-e4b                ok in 1 min
  quality_dev_mistral-7b           ok in 4 min
  quality_heldout_mistral-7b       ok in 5 min
  claims_mistral-7b                ok in 1 min
  quality_dev_phi4-14b             ok in 9 min
  quality_heldout_phi4-14b         ok in 10 min
  claims_phi4-14b                  ok in 1 min
  judge_gemma4-e4b                 ok in 14 min
  judge_report_gemma4-e4b          ok in 0 min
  judge_mistral-7b                 ok in 9 min
  judge_report_mistral-7b          ok in 0 min
  judge_phi4-14b                   ok in 13 min
  judge_report_phi4-14b            ok in 0 min
  judge_ragtruth                   ok in 19 min
  ragtruth_nli_small               ok in 1

In [8]:
# ---- summary: the head of each report -----------------------------------------------------------------------
import glob
for rep in sorted(glob.glob(f"{R}/*/report.md")):
    print("=" * 100, "\n", rep.replace(R + "/", ""), "\n")
    print("".join(open(rep, errors="replace").readlines()[:30]))

 indep_gemma4-e4b/report.md 

# Independent check of Layer 3

300 draft sentences · judge: qwen2.5:14b (local, another model family) · human labels: 0

### Verifier against the judge (all sentences) (300 sentences)

| Measure | Value |
| --- | --- |
| Supported sentences the verifier kept (recall) | 86.2% [82-90] (256/297) |
| Unsupported sentences the verifier released | 66.7% [21-94] (2/3) |
| Released sentences that are supported (precision) | 99.2% [97-100] (256/258) |
| Agreement (supported vs not) | 85.7% [81-89] (257/300) |
| Cohen's kappa (supported vs not) | 0.03 |

### Verifier against the judge, dev questions (132 sentences)

| Measure | Value |
| --- | --- |
| Supported sentences the verifier kept (recall) | 86.9% [80-92] (113/130) |
| Unsupported sentences the verifier released | 50.0% [9-91] (1/2) |
| Released sentences that are supported (precision) | 99.1% [95-100] (113/114) |
| Agreement (supported vs not) | 86.4% [79-91] (114/132) |
| Cohen's kappa (supported vs not) 